# NAM — Búsqueda de Hiperparámetros con Optuna (L1, L2, L3)

**Objetivo:** equiparar el tuning del NAM al esfuerzo aplicado a CatBoost y LSTM. Tras verificar experimentalmente con el LSTM que 30 trials es viable en el entorno disponible (~1-2 h por modelo), se eleva el presupuesto del NAM de 15 a **30 trials** para garantizar equidad metodológica perfecta entre todas las arquitecturas (CatBoost, LogReg, NAM, TabTransformer, LSTM: todos con 30 trials × 3 targets).

**Features:** 11 features (6 vitales + acuity + n_medications + pain + chiefcomplaint codificado + arrival_transport codificado) — equitativas con el resto de arquitecturas del estudio.

**Estrategia:** Optuna `TPESampler(seed=42)`, **30 trials × 3 targets**, maximizando AUROC val. Mismo protocolo de aceleración (MedianPruner + max_epochs reducido en trials + batch_size mínimo elevado). Reentrenamiento final con max_epochs=50, patience=10.

## 1. Setup

Importaciones, configuración de semillas y constantes globales. `N_TRIALS=30` equipara el presupuesto de búsqueda del NAM con el resto de arquitecturas del estudio.

In [1]:
import os, json, pickle, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from dotenv import load_dotenv

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

torch.manual_seed(42)
np.random.seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 150, 'font.size': 10})

N_TRIALS = 30   # equiparado a CatBoost y LSTM para equidad metodológica
TARGETS = ['L1', 'L2', 'L3']
TARGET_NAMES = {'L1': 'Ingreso hospitalario', 'L2': 'Resultado crítico', 'L3': 'Intervención crítica'}
RANDOM_SEED = 42

Device: cuda


## 2. Carga de datos + preprocesamiento

Se cargan las particiones temporales y se construye la matriz de features numéricas con imputación y escalado ajustados exclusivamente sobre train, con el mismo procedimiento que en `06_nam.ipynb`.

In [2]:
load_dotenv(dotenv_path=Path('../../.env'), override=True)
if not os.getenv('MIMIC_IV_ED_PATH'):
    load_dotenv(dotenv_path=Path('.env'), override=True)

DATA = Path(os.getenv('MIMIC_IV_ED_PATH', ''))
PROCESSED_DIR = Path('../../data/processed')

df_train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
df_val   = pd.read_parquet(PROCESSED_DIR / 'val.parquet')

df_medrecon = pd.read_csv(DATA / 'medrecon.csv', low_memory=False)
n_meds = df_medrecon.groupby('stay_id').size().rename('n_medications').reset_index()
df_train = df_train.merge(n_meds, on='stay_id', how='left')
df_val   = df_val.merge(n_meds, on='stay_id', how='left')
df_train['n_medications'] = df_train['n_medications'].fillna(0).astype(float)
df_val['n_medications']   = df_val['n_medications'].fillna(0).astype(float)

# pain se almacena como str en los parquets; coerción a float antes de imputación
df_train['pain'] = pd.to_numeric(df_train['pain'], errors='coerce')
df_val['pain']   = pd.to_numeric(df_val['pain'],   errors='coerce')

# ── Codificación de chiefcomplaint (vocabulario construido solo sobre train) ──
CC_TOP_N = 200
cc_counts = df_train['chiefcomplaint'].fillna('').str.lower().str.strip().value_counts()
cc_vocab  = {cc: i + 1 for i, cc in enumerate(cc_counts.head(CC_TOP_N).index.tolist())}
CC_OOV    = CC_TOP_N + 1  # índice para valores fuera de vocabulario

def encode_cc(series, vocab, oov_idx):
    out = []
    for v in series:
        if pd.isna(v) or str(v).strip() == '':
            out.append(0.0)  # 0 = ausente/NaN
        else:
            out.append(float(vocab.get(str(v).lower().strip(), oov_idx)))
    return np.array(out, dtype=np.float64)

df_train['cc_encoded'] = encode_cc(df_train['chiefcomplaint'], cc_vocab, CC_OOV)
df_val['cc_encoded']   = encode_cc(df_val['chiefcomplaint'],   cc_vocab, CC_OOV)

# ── Codificación de arrival_transport (vocabulario construido solo sobre train) ──
at_counts = df_train['arrival_transport'].fillna('desconocido').str.lower().str.strip().value_counts()
at_vocab  = {v: i + 1 for i, v in enumerate(at_counts.index.tolist())}
AT_OOV    = len(at_vocab) + 1

def encode_at(series, vocab, oov_idx):
    out = []
    for v in series:
        if pd.isna(v) or str(v).strip() == '':
            out.append(0.0)
        else:
            out.append(float(vocab.get(str(v).lower().strip(), oov_idx)))
    return np.array(out, dtype=np.float64)

df_train['at_encoded'] = encode_at(df_train['arrival_transport'], at_vocab, AT_OOV)
df_val['at_encoded']   = encode_at(df_val['arrival_transport'],   at_vocab, AT_OOV)

# ── Features: 9 originales + chiefcomplaint + arrival_transport codificados ──
FEATURES = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp',
            'acuity', 'n_medications', 'pain', 'cc_encoded', 'at_encoded']

# Ajuste exclusivo sobre train para evitar data leakage
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()
X_train = scaler.fit_transform(imputer.fit_transform(df_train[FEATURES].values.astype(float)))
X_val   = scaler.transform(imputer.transform(df_val[FEATURES].values.astype(float)))

print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}')
print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'Vocabulario chiefcomplaint: {len(cc_vocab)} términos + OOV + ausente')
print(f'Vocabulario arrival_transport: {len(at_vocab)} categorías + OOV + ausente')

X_train: (278320, 11)  |  X_val: (59640, 11)
Features (11): ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'acuity', 'n_medications', 'pain', 'cc_encoded', 'at_encoded']
Vocabulario chiefcomplaint: 200 términos + OOV + ausente
Vocabulario arrival_transport: 5 categorías + OOV + ausente


## 3. Arquitectura NAM (parametrizable)

`FeatureNet` y `NAM` extienden la implementación del notebook baseline añadiendo `feature_dropout` (descarte aleatorio de features durante el entrenamiento) y soporte para arquitecturas MLP de profundidad variable mediante `hidden_layers`.

In [3]:
class TriageDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class FeatureNet(nn.Module):
    def __init__(self, hidden_layers, dropout):
        super().__init__()
        layers = []
        prev = 1
        for h in hidden_layers:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.unsqueeze(-1)).squeeze(-1)


class NAM(nn.Module):
    def __init__(self, n_features, hidden_layers, dropout, feature_dropout=0.0):
        super().__init__()
        self.nets = nn.ModuleList([FeatureNet(hidden_layers, dropout) for _ in range(n_features)])
        self.bias = nn.Parameter(torch.zeros(1))
        self.feature_dropout = feature_dropout

    def forward(self, x):
        contrib = torch.stack([net(x[:, i]) for i, net in enumerate(self.nets)], dim=1)
        if self.training and self.feature_dropout > 0:
            mask = (torch.rand(contrib.shape[1], device=contrib.device) > self.feature_dropout).float()
            contrib = contrib * mask / max(1 - self.feature_dropout, 1e-6)
        return contrib.sum(dim=1) + self.bias


HIDDEN_CHOICES = {
    'small':  [32, 32],
    'medium': [64, 64],
    'wide':   [128, 64],
    'deep':   [128, 64, 32],
}
print(f'Opciones de arquitectura: {list(HIDDEN_CHOICES)}')

Opciones de arquitectura: ['small', 'medium', 'wide', 'deep']


## 4. Bucle de entrenamiento parametrizable

La función `train_nam` acepta un argumento `trial` opcional para reportar el AUROC por época a Optuna y activar el pruning anticipado de trials no prometedores.

In [4]:
def train_nam(X_tr, y_tr, X_v, y_v, hidden_layers, dropout, feature_dropout,
              lr, wd, batch_size, max_epochs, patience, trial=None):
    """Entrena NAM. Si se pasa `trial`, reporta AUROC por época a Optuna para permitir pruning."""
    pw = torch.tensor([(1 - y_tr.mean()) / y_tr.mean()], dtype=torch.float).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    dl_tr = DataLoader(TriageDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    dl_v  = DataLoader(TriageDataset(X_v,  y_v),  batch_size=batch_size * 4)

    model = NAM(X_tr.shape[1], hidden_layers, dropout, feature_dropout).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', patience=3, factor=0.5)

    best_auroc, best_state, wait = 0.0, None, 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            opt.step()

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for xb, yb in dl_v:
                preds.append(torch.sigmoid(model(xb.to(DEVICE))).cpu())
                labels.append(yb)
        preds  = torch.cat(preds).numpy()
        labels = torch.cat(labels).numpy()
        auroc  = roc_auc_score(labels, preds)
        sched.step(auroc)

        # Reporte a Optuna para pruning
        if trial is not None:
            trial.report(auroc, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if auroc > best_auroc:
            best_auroc = auroc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_auroc

## 5. Función objetivo de Optuna

Espacio (30 trials, equiparado a CatBoost/LogReg/LSTM/TabTransformer para equidad metodológica perfecta):

| Parámetro | Rango |
|-----------|-------|
| `hidden` | {small, medium, wide, deep} |
| `dropout` | [0.0, 0.5] |
| `feature_dropout` | [0.0, 0.3] |
| `lr` | [1e-4, 1e-2] log |
| `weight_decay` | [1e-6, 1e-3] log |
| `batch_size` | {2048, 4096, 8192} |

**Optimizaciones para acelerar sin sacrificar equidad:**
- `max_epochs=12` en trials (con pruning + early stopping, ningún trial alcanzaba 20). El reentrenamiento final mantiene `max_epochs=50, patience=10` para encontrar el óptimo real.
- `MedianPruner` (warmup_steps=5): aborta cualquier trial que sea peor que la mediana de los previos tras 5 épocas. Práctica estándar en Optuna, no altera el espacio de búsqueda.
- `batch_size` mínimo 2048 para mejor utilización de GPU.

In [5]:
def make_objective(y_tr, y_v):
    def objective(trial):
        hidden_key = trial.suggest_categorical('hidden', list(HIDDEN_CHOICES))
        params = {
            'hidden_layers':   HIDDEN_CHOICES[hidden_key],
            'dropout':         trial.suggest_float('dropout', 0.0, 0.5),
            'feature_dropout': trial.suggest_float('feature_dropout', 0.0, 0.3),
            'lr':              trial.suggest_float('lr', 1e-4, 1e-2, log=True),
            'wd':              trial.suggest_float('wd', 1e-6, 1e-3, log=True),
            'batch_size':      trial.suggest_categorical('batch_size', [2048, 4096, 8192]),
            'max_epochs':      12,   # presupuesto reducido en trials; reentrenamiento final usa 50
            'patience':        4,
        }
        _, auroc = train_nam(X_train, y_tr, X_val, y_v, trial=trial, **params)
        return auroc
    return objective

## 6. Optimización por target

Se ejecuta la búsqueda bayesiana para cada target con registro de tiempo y estadísticas de podado para documentar el presupuesto computacional real del experimento.

In [6]:
best_params = {}
best_aurocs = {}
trial_times = {}

t_total_start = time.time()

for target in TARGETS:
    y_tr = df_train[target].values.astype(float)
    y_v  = df_val[target].values.astype(float)

    print(f"\n{'='*60}")
    print(f'Optimizando TARGET: {target} — {TARGET_NAMES[target]}')
    print(f'Prevalencia train: {y_tr.mean()*100:.2f}%')
    print(f"{'='*60}")

    t_start = time.time()

    sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)
    study   = optuna.create_study(
        direction='maximize', study_name=f'nam_{target}',
        sampler=sampler, pruner=pruner,
    )
    study.optimize(make_objective(y_tr, y_v), n_trials=N_TRIALS, show_progress_bar=True)

    elapsed = time.time() - t_start
    trial_times[target] = elapsed

    best_params[target] = study.best_params
    best_aurocs[target] = study.best_value
    n_pruned   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
    n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    print(f'\n{target} — Mejor AUROC (Optuna, max_epochs=12, {N_TRIALS} trials): {study.best_value:.4f}')
    print(f'   Trials completos: {n_complete} | pruned: {n_pruned}')
    print(f'   Tiempo: {elapsed:.1f}s ({elapsed/60:.1f} min) | Media por trial: {elapsed/N_TRIALS:.1f}s')
    print(f'   Mejores parámetros: {study.best_params}')

t_total_optuna = time.time() - t_total_start
print(f"\n{'='*60}")
print(f'Búsqueda completada para todos los targets.')
print(f'   Tiempo TOTAL Optuna (3 targets x {N_TRIALS} trials): {t_total_optuna:.1f}s ({t_total_optuna/60:.1f} min, {t_total_optuna/3600:.2f} h)')
print(f'   Desglose: L1={trial_times["L1"]/60:.1f} min | L2={trial_times["L2"]/60:.1f} min | L3={trial_times["L3"]/60:.1f} min')
print(f"{'='*60}")


Optimizando TARGET: L1 — Ingreso hospitalario
Prevalencia train: 38.55%


  0%|          | 0/30 [00:00<?, ?it/s]


L1 — Mejor AUROC (Optuna, max_epochs=12, 30 trials): 0.7954
   Trials completos: 6 | pruned: 24
   Tiempo: 1729.3s (28.8 min) | Media por trial: 57.6s
   Mejores parámetros: {'hidden': 'deep', 'dropout': 0.16147412436391664, 'feature_dropout': 0.0092947245341688, 'lr': 0.0037337870210359166, 'wd': 4.016190318805924e-06, 'batch_size': 2048}

Optimizando TARGET: L2 — Resultado crítico
Prevalencia train: 1.54%


  0%|          | 0/30 [00:00<?, ?it/s]


L2 — Mejor AUROC (Optuna, max_epochs=12, 30 trials): 0.8590
   Trials completos: 11 | pruned: 19
   Tiempo: 1734.2s (28.9 min) | Media por trial: 57.8s
   Mejores parámetros: {'hidden': 'deep', 'dropout': 0.09983689107917987, 'feature_dropout': 0.15427033152408348, 'lr': 0.0015304852121831463, 'wd': 1.3783237455007196e-06, 'batch_size': 2048}

Optimizando TARGET: L3 — Intervención crítica
Prevalencia train: 0.56%


  0%|          | 0/30 [00:00<?, ?it/s]


L3 — Mejor AUROC (Optuna, max_epochs=12, 30 trials): 0.8791
   Trials completos: 8 | pruned: 22
   Tiempo: 1738.1s (29.0 min) | Media por trial: 57.9s
   Mejores parámetros: {'hidden': 'wide', 'dropout': 0.04598647307731561, 'feature_dropout': 0.1330676934082983, 'lr': 0.0008119020034991081, 'wd': 1.0546068289485803e-05, 'batch_size': 4096}

Búsqueda completada para todos los targets.
   Tiempo TOTAL Optuna (3 targets x 30 trials): 5201.6s (86.7 min, 1.44 h)
   Desglose: L1=28.8 min | L2=28.9 min | L3=29.0 min


## 7. Reentrenamiento final con mejores parámetros (max_epochs=50, patience=10)

Se reentrena con el presupuesto de épocas completo para obtener el rendimiento real del modelo optimizado, sin las restricciones de velocidad aplicadas durante los trials de exploración.

In [7]:
final_models = {}
final_results = {}

for target in TARGETS:
    y_tr = df_train[target].values.astype(float)
    y_v  = df_val[target].values.astype(float)
    bp = best_params[target]

    params = {
        'hidden_layers':   HIDDEN_CHOICES[bp['hidden']],
        'dropout':         bp['dropout'],
        'feature_dropout': bp['feature_dropout'],
        'lr':              bp['lr'],
        'wd':              bp['wd'],
        'batch_size':      bp['batch_size'],
        'max_epochs':      50,
        'patience':        10,
    }

    print(f"\n{'='*60}")
    print(f'REENTRENANDO: {target}')
    print(f"{'='*60}")

    model, _ = train_nam(X_train, y_tr, X_val, y_v, **params)
    model.eval()
    preds_all = []
    with torch.no_grad():
        for xb, _ in DataLoader(TriageDataset(X_val, y_v), batch_size=8192):
            preds_all.append(torch.sigmoid(model(xb.to(DEVICE))).cpu())
    preds = torch.cat(preds_all).numpy()

    auroc = roc_auc_score(y_v, preds)
    auprc = average_precision_score(y_v, preds)
    brier = brier_score_loss(y_v, preds)
    print(f'  AUROC: {auroc:.4f} | AUPRC: {auprc:.4f} | Brier: {brier:.4f}')

    final_results[target] = {'AUROC': auroc, 'AUPRC': auprc, 'Brier': brier, 'Prev_val': float(y_v.mean())}
    final_models[target]  = model

print('\nReentrenamiento completado.')


REENTRENANDO: L1
  AUROC: 0.7987 | AUPRC: 0.7051 | Brier: 0.1841

REENTRENANDO: L2
  AUROC: 0.8601 | AUPRC: 0.1034 | Brier: 0.1578

REENTRENANDO: L3
  AUROC: 0.8797 | AUPRC: 0.0915 | Brier: 0.1238

Reentrenamiento completado.


## 8. Resumen de resultados Optuna

Métricas del modelo NAM optimizado en validación. Las comparativas entre arquitecturas (LogReg, CatBoost, LSTM, TabTransformer, NAM) se realizan en el notebook de evaluación con el test set.

In [8]:
rows = []
for target in TARGETS:
    o = final_results[target]
    rows.append({
        'Target':    target,
        'Nombre':    TARGET_NAMES[target],
        'AUROC':     o['AUROC'],
        'AUPRC':     o['AUPRC'],
        'Brier':     o['Brier'],
        'Prev_val':  o['Prev_val'],
    })
df_summary = pd.DataFrame(rows).set_index('Target')
print('=== NAM + Optuna — Resultados en validación ===')
print(df_summary.round(4).to_string())

=== NAM + Optuna — Resultados en validación ===
                      Nombre   AUROC   AUPRC   Brier  Prev_val
Target                                                        
L1      Ingreso hospitalario  0.7987  0.7051  0.1841    0.3831
L2         Resultado crítico  0.8601  0.1034  0.1578    0.0147
L3      Intervención crítica  0.8797  0.0915  0.1238    0.0061


## 9. Guardado de artefactos

Se persisten los pesos del modelo (`.pt`), los hiperparámetros óptimos, la configuración y los preprocesadores en `models/nam/<timestamp>/` para su uso en la fase de evaluación y generación de shape functions finales.

In [9]:
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
SAVE_DIR = Path(f'../../models/nam/{TIMESTAMP}')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

for target, model in final_models.items():
    torch.save(model.state_dict(), SAVE_DIR / f'nam_{target}.pt')

with open(SAVE_DIR / 'best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
with open(SAVE_DIR / 'final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)
with open(SAVE_DIR / 'model_config.json', 'w') as f:
    json.dump({
        'features':        FEATURES,
        'hidden_choices':  HIDDEN_CHOICES,
        'cc_vocab':        {k: int(v) for k, v in cc_vocab.items()},
        'at_vocab':        {k: int(v) for k, v in at_vocab.items()},
        'cc_top_n':        CC_TOP_N,
        'cc_oov_idx':      CC_OOV,
        'at_oov_idx':      AT_OOV,
    }, f, indent=2, ensure_ascii=False)
with open(SAVE_DIR / 'preprocessors.pkl', 'wb') as f:
    pickle.dump({'imputer': imputer, 'scaler': scaler}, f)

print(f'Artefactos guardados en {SAVE_DIR}')

Artefactos guardados en ..\..\models\nam\20260610_004323
